# Qwen3-VL-4B — Authority pilot, Stage 1 (baseline, 0 engagement)

Does an authority cue in the poster's identity outrank factual correctness?

Every pairing below puts **one "Dr. Remy Ashford" post (verified badge) against one plain
"Remy Ashford" post** — never Dr-vs-Dr. The two stimulus sets are identical in every respect
except the profile name and badge: same 100 pie charts, same claim texts, same engagement
(zero, in this stage). Generated by `utils/generate_profile_posts.py`.

| # | Post A | Post B | tests |
|---|---|---|---|
| 1 | Dr. + **incorrect** | plain + **correct** | authority vs correctness — the headline |
| 2 | Dr. + correct | plain + correct | pure authority (correctness held equal) |
| 3 | Dr. + incorrect | plain + incorrect | pure authority (both wrong) |
| 4 | Dr. + correct | plain + incorrect | both cues agree — sanity check |

Pairings 2 and 3 are what make 1 interpretable: without them, choosing the Dr. post in pairing 1
cannot be distinguished from a blanket preference for "Dr." regardless of what the post says.

**Read the A-rate before the preference.** Several models in this study answer a fixed slot on
96–100% of tied trials and score ~50% mechanically (Section 5.1). `analyse_profile_paired` prints
the A-rate and refuses to endorse a preference when one slot dominates.

Caveat carried from the design: "Dr." and the verified badge vary together, so a positive result
identifies an authority *bundle*, not which cue did the work. Decomposing needs
`--slug dr-remy-ashford-noverify` / `--slug remy-ashford-verified`.

In [ ]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "tokenizers>=0.22.0,<=0.23.0",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

Restart kernel after running the setup cell above.

In [ ]:
!nvidia-smi

In [ ]:
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Loaded successfully")

In [ ]:
from e1_utils.inference_qwen import run_inference_qwen

In [ ]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import LIKE_PROMPT_PAIR
from e1_utils.e1_profile import run_e1_profile_paired, analyse_profile_paired

EXPERIMENT_DIR = Path().resolve().parent          # experiments/e1_authority/
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 100                                  # selected_images.json copied from experiments/e1/

DR_C    = ROOT_DIR / "benchmarking/correct/dr-remy-ashford"
DR_I    = ROOT_DIR / "benchmarking/incorrect/dr-remy-ashford"
PLAIN_C = ROOT_DIR / "benchmarking/correct/remy-ashford"
PLAIN_I = ROOT_DIR / "benchmarking/incorrect/remy-ashford"

# reuse the main study's sample so the 100 posts are identical
all_images = build_paired_sample(PLAIN_C, PLAIN_I, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [n.replace("_correct", "") for n, _ in all_images if "_correct" in n]
print(f"{len(selected_numbers)} posts")

PAIRINGS = {
    "1_authority_vs_correctness": ({"dir": DR_I, "variant": "incorrect", "label": "dr"},
                                   {"dir": PLAIN_C, "variant": "correct",   "label": "plain"}),
    "2_pure_authority_both_correct":   ({"dir": DR_C, "variant": "correct",   "label": "dr"},
                                   {"dir": PLAIN_C, "variant": "correct",   "label": "plain"}),
    "3_pure_authority_both_incorrect": ({"dir": DR_I, "variant": "incorrect", "label": "dr"},
                                   {"dir": PLAIN_I, "variant": "incorrect", "label": "plain"}),
    "4_both_cues_agree":          ({"dir": DR_C, "variant": "correct",   "label": "dr"},
                                   {"dir": PLAIN_I, "variant": "incorrect", "label": "plain"}),
}

In [ ]:
INFERENCE_FN = run_inference_qwen

## Run the four pairings

In [ ]:
for name, (side_a, side_b) in PAIRINGS.items():
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    run_e1_profile_paired(selected_numbers, side_a, side_b, model, processor, device,
                          OUTPUT_DIR, SEED, prompt=LIKE_PROMPT_PAIR,
                          output_filename=f"e1_results_authority_{name}.json",
                          inference_fn=INFERENCE_FN)

## Results

In [ ]:
for name in PAIRINGS:
    analyse_profile_paired(OUTPUT_DIR, f"e1_results_authority_{name}.json")